In [1]:
! pip install  torch transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion", "split")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [4]:
ds["train"][0]

{'text': 'i didnt feel humiliated', 'label': 0}

In [5]:
import pandas as pd
train_ds = pd.DataFrame(ds["train"].shuffle(seed=42))

validation_ds = pd.DataFrame(ds["validation"].shuffle(seed=42))

test_ds = pd.DataFrame(ds["test"])

In [ ]:
train_ds.head()

,text,label
0,while cycling in the country,4
1,i had pocket qq and was feeling pretty confide...,1
2,i am in no way complaining or whining or feeli...,0
3,i feel a bit stressed because it feels like im...,3
4,i tell the people closest to me things that i ...,5


In [ ]:
train_ds.info()

<class 'pandas.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    16000 non-null  str  
 1   label   16000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 1.7 MB


In [ ]:
train_ds["label"].value_counts()

label
1    5362
0    4666
3    2159
4    1937
2    1304
5     572
Name: count, dtype: int64

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
filter = (train_ds["label"] == 1)
train_ds[filter].head(20)

,text,label
1,i had pocket qq and was feeling pretty confident lol,1
6,i am trying to work on finding the joy in the simple thing that god is finding joy in my obedience to him even if it doesn t feel very joyful in the way that i am used to,1
13,im even starting to feel more sociable,1
15,i feel like things are getting a little overwhelming a few spritz of this toner really helps calm and soothe me,1
27,i feel more content with what i have achieved and i know if i don t write today there ll still be a tomorrow,1
28,i don t want to tell people how my first was with you and how you made me feel i don t want to think that you re the most gorgeous guy i ve ever seen and i love how other people disagree because i don t want them to see how truly wonderful you are to me,1
30,i think what i m going to do is care less about anything that doesn t matter and won t make me feel successful in life,1
32,i met you i used to want to lock myself into a vault just to feel precious,1
33,i feel like this will be an amazing series and will be epic in the movie theater,1
35,i feel it is my solemn duty to share this divine knowledge of mine in order that others may benefit from it s truth and beauty and render their world just a tad closer to thearchitecturality that utopian perfectly set garage society to which we all strive,1


In [ ]:
idx2label = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}

# Preprocessing

In [8]:
X_train = train_ds["text"]
y_train = train_ds["label"]

X_val = validation_ds["text"]
y_val = validation_ds["label"]

X_test = test_ds["text"]
y_test = test_ds["label"]

# Modeling

## BaseLine

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import time

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=(2, 5),
        max_features=50000
    )),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=42,
        n_jobs=-1
    ))
])

In [9]:
start_time = time.time()
pipeline.fit(X_train, y_train)
end_time = time.time()
print(f"Training completed in {end_time - start_time:.2f} seconds.")

Training completed in 9.21 seconds.


In [11]:
from sklearn.metrics import classification_report


train_predictions = pipeline.predict(X_train)


print(classification_report(y_train, train_predictions))

              precision    recall  f1-score   support

           0       0.85      0.97      0.90      4666
           1       0.76      0.99      0.86      5362
           2       0.99      0.40      0.57      1304
           3       0.99      0.73      0.84      2159
           4       0.97      0.72      0.83      1937
           5       0.99      0.19      0.32       572

    accuracy                           0.84     16000
   macro avg       0.92      0.67      0.72     16000
weighted avg       0.87      0.84      0.82     16000



In [10]:
val_predictions = pipeline.predict(X_val)


print(classification_report(y_val, val_predictions))

              precision    recall  f1-score   support

           0       0.59      0.77      0.67       550
           1       0.58      0.88      0.69       704
           2       0.92      0.12      0.22       178
           3       0.91      0.31      0.46       275
           4       0.77      0.30      0.43       212
           5       0.88      0.09      0.16        81

    accuracy                           0.61      2000
   macro avg       0.77      0.41      0.44      2000
weighted avg       0.69      0.61      0.56      2000



## Fine Tuning Bert

In [12]:
import torch
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

print("Downloading Tokenizer")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Ready")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Ready


In [13]:
def tokenize_function(examples):
  return tokenizer(examples["text"], padding="max_length", truncation=True, max_length= 128)

In [14]:
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_ds)
validation_dataset = Dataset.from_pandas(validation_ds)
test_dataset = Dataset.from_pandas(test_ds)

In [15]:
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_validation_dataset = validation_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [18]:
tokenized_train_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 16000
})

In [17]:
tokenized_train_dataset["text"][0]

'while cycling in the country'

In [19]:
tokenized_train_dataset["input_ids"][0]

[101,
 2096,
 9670,
 1999,
 1996,
 2406,
 102,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [21]:
tokenized_train_dataset["attention_mask"][0]

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [27]:
import numpy as np
import evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, IntervalStrategy

In [23]:
NUM_CLASSES = 6

print("Downloading the model")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_CLASSES)

f1_metric = evaluate.load("f1")




model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  predictions = np.argmax(predictions, axis=1)
  return f1_metric.compute(predictions=predictions, references=labels, average="macro")

In [28]:
training_args = TrainingArguments(
    output_dir="./emotion_model_results",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay = 0.01,
    eval_strategy=IntervalStrategy.EPOCH,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none"
)

In [30]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train_dataset,
    eval_dataset = tokenized_validation_dataset,
    compute_metrics = compute_metrics
)

In [31]:
print("FineTuning")
trainer.train()
print("Finished")

FineTuning


Epoch,Training Loss,Validation Loss,F1
1,0.222070,0.216263,0.905706
2,0.187154,0.170999,0.902085
3,0.131141,0.148820,0.905067


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Finished


In [32]:
raw_predictions = trainer.predict(tokenized_validation_dataset)

In [33]:
y_pred = np.argmax(raw_predictions.predictions, axis=1)

In [34]:
y_true = tokenized_validation_dataset["label"]

In [35]:
target_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

In [36]:
print(classification_report(y_true, y_pred, target_names=target_names))

              precision    recall  f1-score   support

     sadness       0.92      0.98      0.95       550
         joy       0.95      0.96      0.95       704
        love       0.91      0.84      0.87       178
       anger       0.96      0.89      0.93       275
        fear       0.86      0.90      0.88       212
    surprise       0.97      0.75      0.85        81

    accuracy                           0.93      2000
   macro avg       0.93      0.89      0.91      2000
weighted avg       0.93      0.93      0.93      2000



In [37]:
trainer.save_model("./emotion_classifier_model")
tokenizer.save_pretrained("./emotion_classifier_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./emotion_classifier_model/tokenizer_config.json',
 './emotion_classifier_model/tokenizer.json')

In [38]:
from transformers import pipeline

emotion_pipeline = pipeline(
    "text-classification",
    model="./emotion_classifier_model",
    tokenizer="./emotion_classifier_model",
    return_all_scores=False
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [39]:
idx2label = {'LABEL_0': 'sadness 😢', 'LABEL_1': 'joy 😊', 'LABEL_2': 'love 🥰',
             'LABEL_3': 'anger 😡', 'LABEL_4': 'fear 😨', 'LABEL_5': 'surprise 😲'}

In [42]:
def predict_user_emotion(text):
    result = emotion_pipeline(text)[0]
    label_id = result['label']
    score = result['score']

    print(f"💬 User text: '{text}'")
    print(f"🔮 Detected Emotion: {idx2label[label_id]} (Confidence: {score*100:.2f}%)")
    print("-" * 50)

In [49]:
import time

start_time = time.time()
predict_user_emotion("I can't believe I passed the exam, this is incredible!")
predict_user_emotion("I feel so lonely and everything is going wrong today.")
predict_user_emotion("Stop messaging me, you are making me so furious.")

print(f"Total time taken: {time.time() - start_time:.2f} seconds")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[{'label': 'LABEL_1', 'score': 0.6015382409095764}]
{'label': 'LABEL_1', 'score': 0.6015382409095764}
💬 User text: 'I can't believe I passed the exam, this is incredible!'
🔮 Detected Emotion: joy 😊 (Confidence: 60.15%)
--------------------------------------------------
[{'label': 'LABEL_0', 'score': 0.9859812259674072}]
{'label': 'LABEL_0', 'score': 0.9859812259674072}
💬 User text: 'I feel so lonely and everything is going wrong today.'
🔮 Detected Emotion: sadness 😢 (Confidence: 98.60%)
--------------------------------------------------
[{'label': 'LABEL_3', 'score': 0.9727634191513062}]
{'label': 'LABEL_3', 'score': 0.9727634191513062}
💬 User text: 'Stop messaging me, you are making me so furious.'
🔮 Detected Emotion: anger 😡 (Confidence: 97.28%)
--------------------------------------------------
Total time taken: 0.07 seconds


In [44]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("./emotion_classifier_model")

id2label = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
label2id = {'sadness': 0, 'joy': 1, 'love': 2, 'anger': 3, 'fear': 4, 'surprise': 5}

config.id2label = id2label
config.label2id = label2id
config.save_pretrained("./emotion_classifier_model")


In [45]:
!zip -r emotion_classifier_model.zip ./emotion_classifier_model

  adding: emotion_classifier_model/ (stored 0%)
  adding: emotion_classifier_model/tokenizer_config.json (deflated 42%)
  adding: emotion_classifier_model/model.safetensors (deflated 8%)
  adding: emotion_classifier_model/training_args.bin (deflated 53%)
  adding: emotion_classifier_model/config.json (deflated 52%)
  adding: emotion_classifier_model/tokenizer.json (deflated 71%)


In [46]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [47]:
!cp emotion_classifier_model.zip /content/drive/MyDrive/bertfinetuning/